# Positive vs. Negative Sentiment Classification

In this notebook, we demonstrate how to interpret a sentiment classification model using SHAP. The goal is to understand how individual words in a movie review influence the model's prediction of positive or negative sentiment.

In [ ]:
import datasets
import numpy as np
import transformers

import shap

## Load the IMDB movie review dataset

In [ ]:
# Load IMDB dataset (movie reviews labeled as positive/negative)
dataset = datasets.load_dataset("imdb", split="test")

# shorten the strings to fit into the pipeline model
short_data = [v[:500] for v in dataset["text"][:20]]

## Load and run a sentiment analysis pipeline

In [ ]:
# Load pretrained sentiment analysis model from HuggingFace
# Note: Model will be downloaded on first run
classifier = transformers.pipeline("sentiment-analysis", return_all_scores=True)
classifier(short_data[:2])

## Explain the sentiment analysis pipeline

In [ ]:
# Create SHAP explainer to compute word-level importance
explainer = shap.Explainer(classifier)

In [ ]:
# explain the predictions of the pipeline on the first two samples
shap_values = explainer(short_data[:2])

In [ ]:
# Visualize contribution of each word to the prediction for the "POSITIVE" class
shap.plots.text(shap_values[:, :, "POSITIVE"])

## Wrap the pipeline manually

SHAP explanations operate best in additive feature spaces. Since probabilities are not additive, we convert them into logits (log-odds). This ensures that SHAP values correctly sum up to the model output.

### Create a TransformersPipeline wrapper

In [ ]:
pmodel = shap.models.TransformersPipeline(classifier, rescale_to_logits=False)

In [ ]:
pmodel(short_data[:2])

In [ ]:
pmodel = shap.models.TransformersPipeline(classifier, rescale_to_logits=True)
pmodel(short_data[:2])

In [ ]:
explainer2 = shap.Explainer(pmodel)
shap_values2 = explainer2(short_data[:2])
shap.plots.text(shap_values2[:, :, 1])

### Pass a tokenizer as the masker object

In [ ]:
explainer2 = shap.Explainer(pmodel, classifier.tokenizer)
shap_values2 = explainer2(short_data[:2])
shap.plots.text(shap_values2[:, :, 1])

### Build a Text masker explicitly

In [ ]:
masker = shap.maskers.Text(classifier.tokenizer)
explainer2 = shap.Explainer(pmodel, masker)
shap_values2 = explainer2(short_data[:2])
shap.plots.text(shap_values2[:, :, 1])

## Explore how the Text masker works

In [ ]:
masker.shape("I like this movie.")

In [ ]:
model_args = masker(np.array([True, True, True, True, True, True, True]), "I like this movie.")
model_args

In [ ]:
pmodel(*model_args)

In [ ]:
model_args = masker(np.array([True, True, False, False, True, True, True]), "I like this movie.")
model_args

In [ ]:
pmodel(*model_args)

In [ ]:
masker2 = shap.maskers.Text(classifier.tokenizer, mask_token="...", collapse_mask_token=True)

In [ ]:
model_args2 = masker2(np.array([True, True, False, False, True, True, True]), "I like this movie.")
model_args2

In [ ]:
pmodel(*model_args2)

## Plot summary statistics and bar charts

In [ ]:
# explain the predictions of the pipeline on the first two samples
shap_values = explainer(short_data[:20])

In [ ]:
shap.plots.bar(shap_values[:, :, "POSITIVE"])

In [ ]:
shap.plots.bar(shap_values[:, :, "POSITIVE"].mean(0))

In [ ]:
shap.plots.bar(shap_values[:, :, "POSITIVE"].mean(0), order=shap.Explanation.argsort)